# Reinforcement learning

In [ ]:
!pip install stable-baselines3[extra]>=2.0.0a4 gymnasium colorama

In [3]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import stable_baselines3 as sb3
from stable_baselines3 import PPO
from stable_baselines3.ppo import MlpPolicy

In [4]:
from gymnasium.utils import seeding

class GridWorldEnv(gym.Env):
    metadata = {"render_modes": ["agent"], "render_fps": 4}

    def __init__(self, grid_size: int, goals_num:int = 1):
        super(GridWorldEnv, self).__init__()
        self.grid_size = grid_size
        # Goals number must be smaller than grid_size^2 to ensure at least one cell is empty
        if goals_num >= grid_size * grid_size:
            raise ValueError(f"Number of goals ({goals_num}) must be less than grid_size^2 ({grid_size * grid_size})")
        self.goals_num = goals_num

        self.action_space = spaces.Discrete(4)  # 0:Up, 1:Right, 2:Down, 3:Left
        self.observation_space = spaces.Box(low=0, high=self.grid_size-1, shape=(2,), dtype=np.int32)

        # Random start position
        self.start_pos = self.np_random.integers(0, self.grid_size, size=(2,), dtype=np.int32)
        self.state = self.start_pos.copy()

        # Generate unique goal positions, avoiding start_pos
        self.goals = []
        while len(self.goals) < self.goals_num:
            goal = self.np_random.integers(0, self.grid_size, size=(2,), dtype=np.int32)
            if not any(np.array_equal(goal, g) for g in self.goals) and not np.array_equal(goal, self.start_pos):
                self.goals.append(goal)

        self.current_goal_index = 0
        self.visited_goals = []

        self.walls = [] 

        while len(self.walls) < self.np_random.integers(1, self.grid_size * self.grid_size // 5):
            wall = tuple(self.np_random.integers(0, self.grid_size, size=(2,)))
            if (
                wall not in self.walls
                and not np.array_equal(wall, self.state)
                and all(not np.array_equal(wall, g) for g in self.goals)
            ):
                self.walls.append(wall)

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self.np_random, _ = seeding.np_random(seed)

        # Random start position
        self.start_pos = self.np_random.integers(0, self.grid_size, size=(2,), dtype=np.int32)
        self.state = self.start_pos.copy()

        # Generate unique goal positions, avoiding start_pos
        self.goals = []
        while len(self.goals) < self.goals_num:
            goal = self.np_random.integers(0, self.grid_size, size=(2,), dtype=np.int32)
            if not any(np.array_equal(goal, g) for g in self.goals) and not np.array_equal(goal, self.start_pos):
                self.goals.append(goal)

        self.current_goal_index = 0
        self.visited_goals = []

        self.walls = [] 

        while len(self.walls) < self.np_random.integers(1, self.grid_size * self.grid_size // 5):
            wall = tuple(self.np_random.integers(0, self.grid_size, size=(2,)))
            if (
                wall not in self.walls
                and not np.array_equal(wall, self.state)
                and all(not np.array_equal(wall, g) for g in self.goals)
            ):
                self.walls.append(wall)
        return self.state.copy(), {}

    def step(self, action):
        x, y = self.state

        if action == 0 and y > 0:     # Up
            y -= 1
        elif action == 1 and x < self.grid_size - 1:  # Right
            x += 1
        elif action == 2 and y < self.grid_size - 1:  # Down
            y += 1
        elif action == 3 and x > 0:   # Left
            x -= 1

        done = False
        reward = -0.1

        new_state = np.array([x, y])
        if tuple(new_state) in self.walls or np.array_equal(new_state, self.state):
            new_state = self.state  # Stay in place if hit a wall
            reward = -1.5

        self.state = new_state

        for i, goal in enumerate(self.goals):
            if np.array_equal(self.state, goal):
                if i == self.current_goal_index:
                    # Correct goal in order
                    reward = 1
                    self.visited_goals.append(self.state.copy())
                    self.current_goal_index += 1
                    if self.current_goal_index >= len(self.goals):
                        done = True  # All goals visited
                elif i in [j for j, g in enumerate(self.goals) if np.array_equal(self.state, g)] and i < self.current_goal_index:
                    # Already visited goal
                    reward = -0.5
                else:
                    # Wrong order
                    reward = -1

                break  # Only one goal can be matched at a time


        return self.state.copy(), reward, done, False, {}

    def render(self):
        grid = np.full((self.grid_size, self.grid_size), ".", dtype="U3")

        # Place goals
        for i, goal in enumerate(self.goals):
            gx, gy = goal
            marker = f"{i+1}"
            grid[gy][gx] = marker

        # Place walls
        for wall in self.walls:
            x, y = wall
            grid[y][x] = "#" # Wall marker

        # Place agent
        x, y = self.state
        grid[y][x] = "A"

        # Print with formatting
        print("\n".join("".join(f"{cell:^3}" for cell in row) for row in grid))
        print(f"Current Goal: {self.current_goal_index + 1} / {len(self.goals)}")
        print(f"Visited Goals: {len(self.visited_goals)}\n")

    def close(self):
        pass


In [5]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env

# Instantiate and check env
env = GridWorldEnv(grid_size=5, goals_num=3)
check_env(env)
env.render()

evaluate_env = GridWorldEnv(grid_size=5, goals_num=3)

 .  1  .  #  # 
 .  .  .  2  . 
 3  .  A  .  . 
 .  .  .  .  # 
 .  .  .  .  . 
Current Goal: 1 / 3
Visited Goals: 0



In [6]:
# Wrap the env if needed
model = PPO("MlpPolicy", env, verbose=1)
model.learn(total_timesteps=int(2e5))
model.save("ppo_gridworld")


Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/app/venvs/An/lib/python3.12/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 157      |
|    ep_rew_mean     | -74.3    |
| time/              |          |
|    fps             | 767      |
|    iterations      | 1        |
|    time_elapsed    | 2        |
|    total_timesteps | 2048     |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 180          |
|    ep_rew_mean          | -80.1        |
| time/                   |              |
|    fps                  | 472          |
|    iterations           | 2            |
|    time_elapsed         | 8            |
|    total_timesteps      | 4096         |
| train/                  |              |
|    approx_kl            | 0.0104980245 |
|    clip_fraction        | 0.107        |
|    clip_range           | 0.2          |
|    entropy_loss         | -1.38        |
|    explained_variance   | -0.0134      |
|    learning_r

In [ ]:
from stable_baselines3.common.evaluation import evaluate_policy

# Random Agent, before training
mean_reward, std_reward = evaluate_policy(model, evaluate_env, n_eval_episodes=100, reward_threshold=0.2, deterministic=True)

print(f"mean_reward:{mean_reward:.2f} +/- {std_reward:.2f}")

/app/venvs/An/lib/python3.12/site-packages/stable_baselines3/common/evaluation.py:70: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


In [8]:
env = GridWorldEnv(grid_size=5, goals_num=3)
model = PPO.load("ppo_gridworld")

obs, _ = env.reset()
for i in range(100):
    action, _ = model.predict(obs)
    obs, reward, done, _, _ = env.step(action)
    env.render()
    if done:
        print("Goal reached! Stop at step:", i)
        break


 .  .  .  .  # 
 .  .  .  .  A 
 #  .  .  .  . 
 3  .  #  .  . 
 1  .  .  2  . 
Current Goal: 1 / 3
Visited Goals: 0

 .  .  .  .  # 
 .  .  .  A  . 
 #  .  .  .  . 
 3  .  #  .  . 
 1  .  .  2  . 
Current Goal: 1 / 3
Visited Goals: 0

 .  .  .  A  # 
 .  .  .  .  . 
 #  .  .  .  . 
 3  .  #  .  . 
 1  .  .  2  . 
Current Goal: 1 / 3
Visited Goals: 0

 .  .  A  .  # 
 .  .  .  .  . 
 #  .  .  .  . 
 3  .  #  .  . 
 1  .  .  2  . 
Current Goal: 1 / 3
Visited Goals: 0

 .  .  .  .  # 
 .  .  A  .  . 
 #  .  .  .  . 
 3  .  #  .  . 
 1  .  .  2  . 
Current Goal: 1 / 3
Visited Goals: 0

 .  .  .  .  # 
 .  .  .  A  . 
 #  .  .  .  . 
 3  .  #  .  . 
 1  .  .  2  . 
Current Goal: 1 / 3
Visited Goals: 0

 .  .  .  A  # 
 .  .  .  .  . 
 #  .  .  .  . 
 3  .  #  .  . 
 1  .  .  2  . 
Current Goal: 1 / 3
Visited Goals: 0

 .  .  .  .  # 
 .  .  .  A  . 
 #  .  .  .  . 
 3  .  #  .  . 
 1  .  .  2  . 
Current Goal: 1 / 3
Visited Goals: 0

 .  .  .  A  # 
 .  .  .  .  . 
 #  .  .  .  . 
 3  .  #